In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt


In [4]:
df = pd.read_csv("/content/drive/MyDrive/new_SOC/Final_data.csv")

soil_gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["Long"], df["Lat"]),
    crs="EPSG:4326"
)


In [9]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt


In [10]:
# Create geometry column
geometry = [Point(xy) for xy in zip(df["Long"], df["Lat"])]

# Create GeoDataFrame
soil_gdf = gpd.GeoDataFrame(df, geometry=geometry)

# Assign CRS (WGS84)
soil_gdf = soil_gdf.set_crs(epsg=4326)

# Show first rows
soil_gdf.head()

,Lat,Long,pH,Ca+++Mg++ (meq/lit.),OC (%),Bulk density (g/cm3),Sand (%),Silt (%),Clay (%),MWD (mm),CO3 (meq/l),HCO3 (meq/l),geometry
0,27.19916,88.56305,6.12,1.8,3.00,1.27,74.76,9.00,16.24,1.17,0.4,0.9,POINT (88.56305 27.19916)
1,26.18472,88.55250,6.22,1.4,2.55,1.38,74.76,15.28,9.96,0.91,0.4,1.2,POINT (88.5525 26.18472)
2,27.19972,88.56861,6.15,1.8,2.62,1.35,72.76,19.48,7.76,1.04,0.5,1.1,POINT (88.56861 27.19972)
3,27.22861,88.56333,6.21,1.8,2.85,1.28,74.76,9.00,16.24,1.06,0.4,1.0,POINT (88.56333 27.22861)
4,27.22472,88.55888,6.15,1.6,2.92,1.28,67.76,10.72,21.52,1.07,0.4,1.1,POINT (88.55888 27.22472)


In [11]:
import seaborn as sns

num_cols = ["OC_%", "pH", "Clay_%", "Sand_%", "Silt_%", "Bulk_density_g/cm3", "Ca+++Mg++_meq/lit.", "MWD_mm", "CO3_meq/l","HCO3_meq/l" ]

corr = soil_gdf[num_cols].corr()

plt.figure(figsize=(6,5))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation matrix of soil properties")
plt.show()

KeyError: "['OC_%', 'Clay_%', 'Sand_%', 'Silt_%', 'Bulk_density_g/cm3', 'Ca+++Mg++_meq/lit.', 'MWD_mm', 'CO3_meq/l', 'HCO3_meq/l'] not in index"

In [ ]:
print("Study area CRS:", study_area.crs)
print("Soil CRS:", soil_gdf.crs)

In [ ]:
soil_gdf_utm = soil_gdf.to_crs(study_area.crs)


In [ ]:
print("Study bounds:", study_area.total_bounds)
print("Soil bounds:", soil_gdf_utm.total_bounds)


In [ ]:
study_area = gpd.read_file("/content/drive/MyDrive/new_SOC/Shape/watershed.shp")
print(study_area.crs)
print(study_area.total_bounds)


In [ ]:
study_area_utm = study_area.to_crs("EPSG:32645")
print(study_area_utm.crs)
print(study_area_utm.total_bounds)


In [ ]:
soil_gdf_utm = soil_gdf.to_crs("EPSG:32645")


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 9))

study_area_utm.plot(
    ax=ax,
    facecolor="#00ffff",
    edgecolor="black",
    alpha=0.6
)

soil_gdf_utm.plot(
    ax=ax,
    color="black",
    markersize=30,
    zorder=5
)

# Zoom
minx, miny, maxx, maxy = study_area_utm.total_bounds
ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)

ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.set_title("Soil Sampling Points within Study Area (UTM Zone 45N)")
ax.set_aspect("equal")

plt.show()


In [ ]:
import numpy as np
import rasterio
from rasterio.features import geometry_mask
from rasterio.transform import from_origin


In [ ]:
# Resolution (use same as covariates, e.g. 30 m)
res = 30

# Study area bounds (UTM)
minx, miny, maxx, maxy = study_area_utm.total_bounds

# Raster size
width = int((maxx - minx) / res)
height = int((maxy - miny) / res)

# Raster transform
transform = from_origin(minx, maxy, res, res)


In [ ]:
mask = geometry_mask(
    study_area_utm.geometry,
    out_shape=(height, width),
    transform=transform,
    invert=True  # True = inside polygon
)


In [ ]:
import rasterio
import numpy as np



In [ ]:
covariate_files = {
    "elevation": "/content/drive/MyDrive/new_SOC/Covariates/Elevation.tif",
    "slope": "/content/drive/MyDrive/new_SOC/Covariates/Slope.tif",
    "twi": "/content/drive/MyDrive/new_SOC/Covariates/Topographic Wetness Index.tif",
    "aspect": "/content/drive/MyDrive/new_SOC/Covariates/Aspect.tif",
    "flow_accumulation": "/content/drive/MyDrive/new_SOC/Covariates/Flow Accumulation.tif",
    "plane_curvature": "/content/drive/MyDrive/new_SOC/Covariates/Plan Curvature.tif",
    "profile_curvature": "/content/drive/MyDrive/new_SOC/Covariates/Profile Curvature.tif",
    "spi": "/content/drive/MyDrive/new_SOC/Covariates/Stream Power Index.tif"
}


In [ ]:
stack = []

for path in covariate_files.values():
    with rasterio.open(path) as src:
        band = src.read(1)
        stack.append(band)

stack = np.stack(stack, axis=-1)  # shape: (H, W, n_features)


In [ ]:
import rasterio
from rasterio.features import geometry_mask
import numpy as np



In [ ]:
ref_path = "/content/drive/MyDrive/new_SOC/Covariates/Elevation.tif"

with rasterio.open(ref_path) as ref:
    ref_transform = ref.transform
    ref_shape = (ref.height, ref.width)
    ref_crs = ref.crs


In [ ]:
mask = geometry_mask(
    study_area_utm.geometry,
    out_shape=ref_shape,
    transform=ref_transform,
    invert=True
)


In [ ]:
covariate_files = [
    "/content/drive/MyDrive/new_SOC/Covariates/Elevation.tif",
    "/content/drive/MyDrive/new_SOC/Covariates/Slope.tif",
    "/content/drive/MyDrive/new_SOC/Covariates/Topographic Wetness Index.tif",
    "/content/drive/MyDrive/new_SOC/Covariates/Aspect.tif",
    "/content/drive/MyDrive/new_SOC/Covariates/Flow Accumulation.tif",
    "/content/drive/MyDrive/new_SOC/Covariates/Plan Curvature.tif",
    "/content/drive/MyDrive/new_SOC/Covariates/Profile Curvature.tif",
    "/content/drive/MyDrive/new_SOC/Covariates/Stream Power Index.tif"
]

stack = []

for path in covariate_files:
    with rasterio.open(path) as src:
        band = src.read(1)
        stack.append(band)

stack = np.stack(stack, axis=-1)  # (H, W, n_features)


In [ ]:
X_pred = stack[mask]   # shape: (n_pixels, n_features)


In [ ]:
print("Raster shape:", stack.shape[:2])
print("Mask shape:", mask.shape)
print("Pixels inside:", mask.sum())


In [ ]:
import numpy as np

# Known NoData encodings
nodata_vals = [65535, 255, 254]

X_pred_clean = np.where(
    np.isin(X_pred, nodata_vals),
    np.nan,
    X_pred
)

# Keep only rows without NaN
valid_pixels = ~np.isnan(X_pred_clean).any(axis=1)
X_pred_final = X_pred_clean[valid_pixels]

print("Pixels for prediction:", X_pred_final.shape[0])


In [ ]:
globals().keys()


In [ ]:
features = [
    "elevation", "slope", "twi", "aspect",
    "flow_accumulation", "plane_curvature",
    "profile_curvature", "spi"
]


In [ ]:
import numpy as np

nodata_vals = [65535, 255, 254]

soil_gdf_clean = soil_gdf_utm.copy()

# Replace raster NoData with NaN
soil_gdf_clean[features] = soil_gdf_clean[features].replace(nodata_vals, np.nan)

# Drop rows with missing predictors or SOC
soil_gdf_clean = soil_gdf_clean.dropna(
    subset=features + ["OC_%"]
)

print("Clean samples:", soil_gdf_clean.shape[0])


In [ ]:
X_train = soil_gdf_clean[features].values
y_train = soil_gdf_clean["OC_%"].values

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)


In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)


In [ ]:
soc_pred = model.predict(X_pred_final)
print("Predictions created:", soc_pred.shape)


In [ ]:
print("Raster prediction pixels:", X_pred_final.shape[0])
print("SOC predictions:", soc_pred.shape[0])


In [ ]:
import numpy as np

soc_raster = np.full(mask.shape, np.nan)

# Fill only valid pixels inside study area
soc_raster[mask] = np.nan
soc_raster[mask][valid_pixels] = soc_pred

print("SOC raster range:",
      np.nanmin(soc_raster),
      np.nanmax(soc_raster))


In [ ]:
import numpy as np

soc_raster = np.full(mask.shape, np.nan)

# Get flat indices of valid pixels inside mask
mask_indices = np.where(mask)

# Apply valid_pixels filter
valid_indices = (
    mask_indices[0][valid_pixels],
    mask_indices[1][valid_pixels]
)

# Assign predictions correctly
soc_raster[valid_indices] = soc_pred

print(
    "SOC raster range:",
    np.nanmin(soc_raster),
    np.nanmax(soc_raster)
)


In [ ]:
X_pred_clean   # shape: (n_pixels, n_features), with NaN


In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X_pred_filled = imputer.fit_transform(X_pred_clean)

print("Any NaN after fill:", np.isnan(X_pred_filled).any())


In [ ]:
soc_pred_full = model.predict(X_pred_filled)

print("Full predictions:", soc_pred_full.shape)


In [ ]:
mask.sum()


In [ ]:
soc_raster_full = np.full(mask.shape, np.nan)

rows, cols = np.where(mask)

soc_raster_full[rows, cols] = soc_pred_full


In [ ]:
plt.figure(figsize=(8, 5))
plt.imshow(soc_raster_full, cmap="YlGn")
plt.colorbar(label="SOC (%)")
plt.title("SOC Prediction Map")
plt.axis("off")
plt.show()


In [ ]:
soc_raster_full


In [ ]:
import numpy as np

# Extract valid SOC values (inside study area only)
valid_soc = soc_raster_full[~np.isnan(soc_raster_full)]

# Quantile breaks (5 classes)
q = np.quantile(valid_soc, [0, 0.2, 0.4, 0.6, 0.8, 1.0])

# Create class raster
soc_class_full = np.full(soc_raster_full.shape, np.nan)

for i in range(5):
    soc_class_full[
        (soc_raster_full >= q[i]) &
        (soc_raster_full < q[i + 1])
    ] = i + 1


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D

# Color map
cmap = ListedColormap([
    "#ffffcc",  # Very Low
    "#c2e699",  # Low
    "#78c679",  # Moderate
    "#31a354",  # High
    "#006837"   # Very High
])

# Get real spatial extent from study area
minx, miny, maxx, maxy = study_area_utm.total_bounds

fig, ax = plt.subplots(figsize=(8, 6))

ax.imshow(
    soc_class_full,
    cmap=cmap,
    extent=[minx, maxx, miny, maxy],
    origin="upper"
)

ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.set_title("SOC Classes (Quantile-based)")
ax.set_aspect("equal")

# Legend
legend_elements = [
    Line2D([0], [0], marker='s', color='w', label='Very Low',
           markerfacecolor=cmap(0), markersize=10),
    Line2D([0], [0], marker='s', color='w', label='Low',
           markerfacecolor=cmap(1), markersize=10),
    Line2D([0], [0], marker='s', color='w', label='Moderate',
           markerfacecolor=cmap(2), markersize=10),
    Line2D([0], [0], marker='s', color='w', label='High',
           markerfacecolor=cmap(3), markersize=10),
    Line2D([0], [0], marker='s', color='w', label='Very High',
           markerfacecolor=cmap(4), markersize=10)
]

ax.legend(handles=legend_elements, loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

mapping_predictors = [
    "elevation",
    "slope",
    "twi",
    "aspect",
    "flow_accumulation",
    "NDVI",
    "SAVI",
    "EVI",
    "NDMI",
    "NDBI",
    "BSI"
]

X_map = soil_gdf[mapping_predictors]
y_map = soil_gdf["OC_%"]

gb_final = GradientBoostingRegressor(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=3,
    subsample=0.8,
    random_state=42
)

gb_final.fit(X_map, y_map)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

importances = gb_final.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(8,6))
plt.bar(range(len(importances)), importances[indices])
plt.xticks(range(len(importances)),
           np.array(mapping_predictors)[indices],
           rotation=45, ha="right")

plt.ylabel("Relative importance")
plt.title("Gradient Boosting Feature Importance")
plt.tight_layout()
plt.show()

fig.savefig("Gradient boosting Feature Importance.png", dpi=300, bbox_inches="tight")
print("Saved!")


In [ ]:
!pip install shap


In [ ]:
import shap


In [ ]:
# Use TreeExplainer for GB model
explainer = shap.TreeExplainer(gb_final)

# SHAP values for soil points
shap_values = explainer.shap_values(X_map)


In [ ]:
shap.summary_plot(
    shap_values,
    X_map,
    plot_type="dot",
    show=True
)



In [ ]:
shap.summary_plot(
    shap_values,
    X_map,
    plot_type="bar",
    show=True
)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

labels = ["RMSE", "MAE", "R²"]
rf_values = [0.370, 0.249, 0.724]
gb_values = [0.366, 0.255, 0.742]

angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
angles += angles[:1]
rf_values += rf_values[:1]
gb_values += gb_values[:1]

fig, ax = plt.subplots(figsize=(7,7), subplot_kw=dict(polar=True))

# Thicker lines added here
ax.plot(angles, rf_values, linewidth=3.5, label="Random Forest", marker="o")
ax.plot(angles, gb_values, linewidth=3.5, label="Gradient Boosting", marker="o")

ax.fill(angles, rf_values, alpha=0.15)
ax.fill(angles, gb_values, alpha=0.15)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=12)
ax.set_title("Model Performance Profile Comparison", pad=25, fontsize=14)
ax.legend(loc="upper right", fontsize=11, framealpha=0.3)

plt.tight_layout()
plt.show()

fig.savefig("SOC_spider_plot.png", dpi=300, bbox_inches="tight")
print("Saved!")